In [ ]:
import environment

# **Notebook**
**Week 1**
**Vraag:** Kunnen we een simulatie maken waarin cellen ontstaan, profileren, communiceren en reageren indien een tumor aanwezig is.

We kunnen de algemene mutatie rate nemen om te zien wanneer een mutatie zou kunnen onstaan en aan de hand hiervan bepalen we dat deze mutatie eigenlijk altijd een tumor mutatie zal zijn.


De cell zal 3 mogelijk statussen hebben.
0: geen cell aanwezig
1: gezonde cell
2: tumor cell

De reden hiervoor is zodat de cell communicatie makkelijk gemoduleerd kan worden. Cellen communiceren met elkaar door signalling pathways. Deze bestaan uit chemische signalen zoals hormonen en eiwitten die door receptoren opgevangen worden. De cellen hebben verschillende soorten contact.

Direct contact is door cellen die strak tegen elkaar aanliggen en communiceren door kanalen die door de celwand heen gaan.

We kunnen door middel van een GUI met bijvoorbeeld tkinter een enorm grid maken gebaseerd op onze gewenste cell grote.

Neem aan dat we een "biopt" hebben met een grote van 50 x 50 pixels, hiervan kunnen we een grid maken waarin we elke column en row kunnen aanroepen. Hiermee kunnen we elke grid 1 visuele pixel aanwijzen en een kleur geven. Eerder benoemd hebben we elk cell type een nummer gegeven. Dit betekent dat we ook elk nummer een kleur kunnen geven.

Er zijn een paar belangerijke dingen om vast te stellen. Hoe modeleer je de cell omgeving? hoe prolifeerd een tumor cell, welke soort kanker is het meest realistisch om te gebruiken? Hoe willen we dat cellen met elkaar communiceren en nog wat andere vragen en problemen die tijdens het process duidelijk zullen worden.

De NCI - National Cancer Institute (_Common cancer types_.) - zegt de volgende top 5 voor de meest voorkomende kankers: borst- en prostaatkanker op nummer 1, gevolgd door long- en colontumor. Het zal dus 1 van deze 4 worden.

---

We kiezen de Agent-Based Modelling-approach en niet de ODE-weg, maar we houden toch wat inspiratie van de ODE om de regels te schrijven voor de ABM. We nemen als inspiratie het Gompertz-model aan. Dit model zegt dat tumorcellen langzaam beginnen te groeien, waarna er een stijging is, des te meer cellen aanwezig zijn, en deze stijging sluit ook weer af, waardoor een soort S-curve ontstaat. (Yin et al., 2019) (Srinivasa et al., 2020).

We moeten denken aan de staten van de cellen, zoals eerder genoemd kunnen de cellen 3 staten hebben. 0 (lege space), 1 (gezond), 2 (tumor).

Een lege stuurt geen signalen, maar kan wel een signaal ontvangen om te veranderen naar een gezonde cell of tumorcel.

Een gezonde cell kan signalen sturen naar lege cellen, en signalen ontvangen van tumor cellen om hun staat te veranderen

Een tumorcel kan signalen versturen naar gezonde cellen en lege cellen om ze te veranderen of te laten groeien tot een tumorcel.

(Misschien laten nog chemo inplannen die tumorcellen dood of veranderd naar een gezonde cell.)

---

De grid heeft een klein probleem. Laten we zeggen dat we een grid hebben van 50 bij 50 en er is op positie 24 een tumorcel. Dit kan zeggen dat de cel 25 hierdoor ook een tumorcel is. waardoor als we cel 25 lezen ook als een tumorcel de 26 cell die gezond is ook zal veranderen waardoor we onrealistisch worden. Hierdoor hebben we 2 grids nodig. De oude en de nieuwe.

Dit kunnen we doen als volgt.
```python
def create_grid(rows, cols):
    grid = [[0 for _ in range(cols)] for _ in range(rows)]
    return grid

def copy_grid(grid):
    copied_grid = [row[:] for row in grid]
    return copied_grid

```

---

Maken we een "thin-agent" of een "fat-agent"



We kunnen nu doormiddel van de aangegeven grote van de grid een tumor plaatsen in de grid.
```python
def create_grid(rows, cols):
    grid = [[0 for _ in range(cols)] for _ in range(rows)]
    return grid

def copy_grid(grid):
    copied_grid = [row[:] for row in grid]
    return copied_grid

def place_tumor(grid, rows, cols):
    row = random.randint(0, rows - 1)
    col = random.randint(0, cols - 1)
    grid[row][col] = 2
    return grid
```


---
We moeten kijken hoe we de tijdstappen maken voor de simulatie

De timeloop moet elke cell checken voor welke status hij bevat 0, 1 of 2 en aan de hand daar van moet de cell communicatie aangeroepen worden.

Als we kijken naar de environment file hebben we de algemene logica voor de verspreiding van de tumor gemaakt.

De tumor zal kijken naar de cellen naast zich om te checken of ze gezond zijn, al een tumor zijn of dat er geen cellen aanwezig zijn.

De tumor zal verspreiden als de cell een status van 1 of 0 heeft maar zal de cell overslaan indien de cell all een tumor cell is.

Het mechanisme is op dit moment een while loop met een nested for loop wat betekent dat de big O hoog is. Later zal optimalisatie nodig hebben echter is het werkend krijgen opdit moment de prioriteit.

---

Tumors zullen niet gegarandeerd elke tijdstap verspreiden als we ons Gompertz model aanhouden. Dit betekent dat we een P_INFECTION zullen opzetten wat de infectie probability (kans) zal opzetten.

# Week 2

Terug komende op hoe de tumor verspreid. Een tumor "duwt" andere cellen uit de weg (Jain et al., 2014). In mijn model lijkt het nu of de cellen een soort "geinfecteerd" worden waardoor ze naar een tumor cell veranderen. Het moet dus eigenlijk aangepast worden om te laten zien dat de cellen een soort weggeduwt worden door de tumor?

We kunnen zeggen dat in onze matrix de tumor de cell zal "overnemen" en dat dit de "Duw" is zoals beschreven in (Jain et al., 2014). Dit moet wel duidelijk vermeld worden in de readme.

Nu dat we weten dat de tumor de andere cellen uit de weg duwt moeten we wel kijken hoe vaak dit gebeurt per tijd stap. Een soort tumor groei kans per tijdstap.

Het is tijd om een type kanker uit te zoeken omdat we graag de formule of wel groei snelheid willen vaststellen. Aan de hand van het eerdere onderzoek hebben vast gesteld dat long kanker een type kanker is wat vaak voorkomt. Ik ga in de long kanker voor non-small lung cancer omdat dit binnen de long kankers het meest voorkomende type is. (Lung Cancer Foundation of America, 2024)

We willen nu kijken wat de doubling time is van de non small lung cancer hiermee kunnen we berekenen hoe we onze "verspreiding" van onze tumor gaan modeleren.

Volgens Jiang et al., 2024, is de verdubbeling tijd van onder andere non small lung cancer median 207 dagen is. Zoals we weten uit de gompertz is dat de tumor initieel heel sloom zal groeien, maar meer tumor cellen betekent meer verspreiding. De formule die we gaan gebruiken is als volgt

(1 - P)^207 = 0.5 ook wel P = 1 - (0.5 ** (1/207)) in python dit geeft ons het volgende getal: 0.003342937006.
Dit getal betekent biologisch de kans dat de tumor zich zal delen per dag is 0.33%.

We modelleren celdeling en expansie als 1 gecombineerd proces waarbij een tumorcel direct een buurcel overneemt. Het fysiek verplaatsen van cellen zou computationeel veel duurder zijn zonder de biologische accuraatheid te verbeteren. Dit is dus een bewuste modelaanname.

In deze simulatie willen we ook dat de gezonde cellen groeien natuurlijk, omdat deze simulatie ook een normale proliferatie heeft. Dit kunnen we doen op dezelefde manier door te kijken naar de dubbeling tijd van long weefsel en met dezelfde berekening als bij de tumor ver dubbeling tijd te doen. Het is long weefsel omdat we eerder vast gesteld hebben dat het een long tumor is.

Normale gezond long weefsel in volwassenen groeien/verdubbelen niet, dus inplaats hier van nemen we aan voor deze simulatie dat de long cellen slow dividing cells zijn zoals lever cellen. We weten dat lever cellen zich zelf regeneren en dat longcellen dit niet doen maar om de simulatie interesanter te maken maken we deze verandering voor de model keuze.

Levers zijn voornamelijk opgemaakt uit hepatocyten ongeveer 80% (Kalra et al., 2023), Deze cellen verdubbelen ongeveer elke 30-49 uur maar laten we voor onze simulatie elke twee dagen aan nemen. (Levy et al., 2015)

Dit geeft volgende formule

(1 - P)^2 = 0.5 dus P = 1 - (0.5 ** (1/2)) = 0.2928932188 dit geeft ons de C_DOUBLING_TIME = 0.2928932188

# References
_Common cancer types_. (2025, May 6). Cancer.gov. https://www.cancer.gov/types/common-cancers

Yin, A., Moes, D. J. A., Van Hasselt, J. G., Swen, J. J., & Guchelaar, H. (2019). A review of mathematical models for tumor dynamics and treatment resistance evolution of solid tumors. _CPT Pharmacometrics & Systems Pharmacology_, _8_(10), 720–737. https://doi.org/10.1002/psp4.12450

Srinivasa, K. G., Siddesh, G. M., & Manisekhar, S. R. (Eds.). (2020). _Statistical modelling and machine learning principles for bioinformatics: Techniques, tools, and applications_. Springer Singapore. https://doi.org/10.1007/978-981-15-8640-8

Jain, R. K., Martin, J. D., & Stylianopoulos, T. (2014). The Role of Mechanical Forces in Tumor Growth and Therapy. Annual Review Of Biomedical Engineering, 16(1), 321–346. https://doi.org/10.1146/annurev-bioeng-071813-105259

Lung Cancer Foundation of America. (2024, 13 januari). Types of Lung Cancer | Lung Cancer Foundation of America. LCFA. https://lcfamerica.org/about-lung-cancer/diagnosis/types/

Jiang, B., Han, D., van der Aalst, C. M., Lancaster, H. L., Vonder, M., Gratama, J.-W. C., Silva, M., Field, J. K., de Koning, H. J., Heuvelmans, M. A., & Oudkerk, M. (2024). Lung cancer volume doubling time by computed tomography: A systematic review and meta-analysis. European Journal of Cancer, 212, Article 114339. https://doi.org/10.1016/j.ejca.2024.114339

Kalra, A., Yetiskul, E., Wehrle, C. J., & Tuma, F. (2023, 1 mei). Physiology, liver. StatPearls - NCBI Bookshelf. https://www.ncbi.nlm.nih.gov/books/NBK535438/

Levy, G., Bomze, D., Heinz, S., Ramachandran, S. D., Noerenberg, A., Cohen, M., Shibolet, O., Sklan, E., Braspenning, J., & Nahmias, Y. (2015). Long-term culture and expansion of primary human hepatocytes. Nature Biotechnology, 33(12), 1264–1271. https://doi.org/10.1038/nbt.3377